In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/nhung-96/super-resolution.git

Cloning into 'super-resolution'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 63 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 1.63 MiB | 17.61 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [3]:
!cat /content/super-resolution/vietnamese_data_set/import_vivos_data.py

from google.colab import files

# 1. Upload file kaggle.json
print("Hãy tải lên file kaggle.json của bạn:")
files.upload()

# 2. Thiết lập môi trường Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 3. Tải dataset VIVOS (Nhanh hơn tải thủ công rất nhiều)
!kaggle datasets download -d kynthesis/vivos-vietnamese-speech-corpus-for-asr

# 4. Giải nén
!unzip -q vivos-vietnamese-speech-corpus-for-asr.zip
print("\n✅ Tải và giải nén VIVOS thành công!")


In [4]:
import os
import json

# Input Kaggle Username and API Key to download data
kaggle_data = {"username":"YOUR_KAGGLE_USERNAME","key":"YOUR_KAGGLE_KEY"}

!mkdir -p ~/.kaggle
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump(kaggle_data, f)

!chmod 600 ~/.kaggle/kaggle.json

# Download dataset
print("Downloading VIVOS...")
!kaggle datasets download -d kynthesis/vivos-vietnamese-speech-corpus-for-asr

# Decompressing
print("Đang giải nén...")
!unzip -q vivos-vietnamese-speech-corpus-for-asr.zip -d vivos_data

# Check
!ls -lh vivos_data

Dataset URL: https://www.kaggle.com/datasets/kynthesis/vivos-vietnamese-speech-corpus-for-asr
License(s): Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)
100% 1.37G/1.37G [01:38<00:00, 14.9MB/s]

Đang giải nén...
total 4.0K
drwxr-xr-x 4 root root 4.0K Apr 20 06:43 vivos


In [5]:
import os
import librosa
import soundfile as sf
from tqdm import tqdm
import glob
# Setup path
INPUT_PATTERN = "/content/vivos_data/vivos/train/waves/*/*.wav"
OUTPUT_DIR = "/content/vivos_8kHz_train"
TARGET_SR = 8000
MAX_FILES = 2000

os.makedirs(OUTPUT_DIR, exist_ok=True)
all_files = glob.glob(INPUT_PATTERN)
files_to_process = all_files[:MAX_FILES]

print(f"Find out {len(all_files)}. Start to downsampling {len(files_to_process)} file...")

for file_path in tqdm(files_to_process, desc="Processing"):
    filename = os.path.basename(file_path)
    speaker_id = os.path.basename(os.path.dirname(file_path))
    new_filename = f"{speaker_id}_{filename}"
    out_path = os.path.join(OUTPUT_DIR, new_filename)

    try:
        y, sr = librosa.load(file_path, sr=None)
        y_8k = librosa.resample(y=y, orig_sr=sr, target_sr=TARGET_SR)
        sf.write(out_path, y_8k, TARGET_SR)
    except Exception as e:
        pass

print(f"\nDone: {OUTPUT_DIR}")

Find out 11660. Start to downsampling 2000 file...


Processing: 100%|██████████| 2000/2000 [00:22<00:00, 90.88it/s] 


Done: /content/vivos_8kHz_train


In [6]:
import torch
import torch.nn as nn

# Residual block for 1D CNN
class ResidualBlock(nn.Module):
    def __init__(self):
        super(ResidualBlock, self).__init__()
        # First convolutional layer
        self.conv1 = nn.Conv1d(64, 64, kernel_size=5, stride=1, padding=2)
        self.Relu1 = nn.ReLU()
        # Second convolutional layer
        self.conv2 = nn.Conv1d(64, 64, kernel_size=5, stride=1, padding=2)

    # Forward pass with skip connection
    def forward(self, x):
        residual = self.conv1(x)
        out = self.Relu1(residual)
        out = self.conv2(out)
        # Add original input to the output (Residual connection)
        return x + out

# Main Network Architecture
class MainNetwork(nn.Module):
    def __init__(self):
        super(MainNetwork, self).__init__()
        # Initial feature extraction layer
        self.conv1M = nn.Conv1d(1, 64, kernel_size=15, padding=7)
        self.reluM = nn.ReLU()

        # Cascade of 5 residual blocks
        self.residual_block1 = ResidualBlock()
        self.residual_block2 = ResidualBlock()
        self.residual_block3 = ResidualBlock()
        self.residual_block4 = ResidualBlock()
        self.residual_block5 = ResidualBlock()

        # Final reconstruction layer
        self.conv2M = nn.Conv1d(64, 1, kernel_size=7, padding=3)

    # Forward pass through the entire network
    def forward(self, x):
        initial_input = x

        # Extract features
        out = self.conv1M(x)
        out = self.reluM(out)

        # Pass through residual blocks
        out = self.residual_block1(out)
        out = self.residual_block2(out)
        out = self.residual_block3(out)
        out = self.residual_block4(out)
        out = self.residual_block5(out)

        # Reconstruct high-resolution audio
        out = self.conv2M(out)

        # Global skip connection
        return initial_input + out

In [7]:
import os
import torch
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader

# Dataset class for audio super-resolution
class AudioDataset(Dataset):
    def __init__(self, lq_dir, hq_dir, target_length=22050):
        self.lq_dir = lq_dir
        self.hq_dir = hq_dir
        self.target_length = target_length
        self.file_names = [f for f in os.listdir(lq_dir) if f.endswith('.wav')]

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        lq_filename = self.file_names[idx]

        # Map LQ to HQ file path
        speaker_id = lq_filename.split('_')[0]
        hq_filename = lq_filename.replace(speaker_id + "_", "", 1)

        lq_path = os.path.join(self.lq_dir, lq_filename)
        hq_path = os.path.join(self.hq_dir, speaker_id, hq_filename)

        # Read audio and sample rate
        lq_waveform, lq_sr = torchaudio.load(lq_path)
        hq_waveform, hq_sr = torchaudio.load(hq_path)

        # Convert to Mono
        if lq_waveform.shape[0] > 1: lq_waveform = lq_waveform[0:1, :]
        if hq_waveform.shape[0] > 1: hq_waveform = hq_waveform[0:1, :]

        # Match sample rates (Upsample LQ to HQ)
        if lq_sr != hq_sr:
            resampler = T.Resample(lq_sr, hq_sr)
            lq_waveform = resampler(lq_waveform)

        # Align exact lengths
        min_len = min(lq_waveform.shape[1], hq_waveform.shape[1])
        lq_waveform = lq_waveform[:, :min_len]
        hq_waveform = hq_waveform[:, :min_len]

        # Trim or pad to target length
        if min_len > self.target_length:
            lq_waveform = lq_waveform[:, :self.target_length]
            hq_waveform = hq_waveform[:, :self.target_length]
        else:
            pad_amount = self.target_length - min_len
            lq_waveform = torch.nn.functional.pad(lq_waveform, (0, pad_amount))
            hq_waveform = torch.nn.functional.pad(hq_waveform, (0, pad_amount))

        return lq_waveform, hq_waveform

# Initialize dataset
train_dataset = AudioDataset(
    lq_dir='/content/vivos_8kHz_train',
    hq_dir='/content/vivos_data/vivos/train/waves'
)

# Initialize dataloader
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
print(f"Done: {len(train_dataset)} segments ready.")

Done: 2000 segments ready.


In [8]:
import torch.optim as optim

# Open GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang chạy trên: {device}")

# Upload model
model = MainNetwork().to(device)
criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 100
best_loss = float('inf')
save_dir = '/content/drive/MyDrive/Audio_SR_Project'
os.makedirs(save_dir, exist_ok=True)

model_save_path = os.path.join(save_dir, 'best_audio_model.pth')

model.train()
print("Start training...")

for epoch in range(num_epochs):
    epoch_loss = 0.0

    for batch_lq, batch_hq in train_loader:
        batch_lq = batch_lq.to(device)
        batch_hq = batch_hq.to(device)

        optimizer.zero_grad()
        outputs = model(batch_lq)
        loss = criterion(outputs, batch_hq)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(train_loader)

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), model_save_path)
        print(f"Epoch {epoch+1:03d}/{num_epochs} | New record Loss: {best_loss:.4f}")
    elif (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:03d}/{num_epochs} | Loss: {avg_loss:.4f}")

Đang chạy trên: cuda
Start training...


KeyboardInterrupt: 